# Lab - API Errors: Fix Yours, Retry the Weather

Three tasks that make requests fail on purpose and read what the library says. An assistant that crashes on the first bad key, or retries a hopeless request forever, is worse than no assistant.

| Task | What you learn |
| --- | --- |
| 1 | Two families: a status error has a code, a connection error has none |
| 2 | Fail three ways - wrong key, removed parameter, a 50 ms timeout |
| 3 | Retry only what can change - and let max_retries do it for you |

**You do not need an API key.** This lab ships with a Claude simulator that fails the way the API does: an explicit key it cannot accept answers 401, a removed parameter 400, a timeout shorter than any reply times out, and `simulate_overload()` makes the next requests answer 529. The code you write is exactly the code you would write against the real API.

**You do not write code from scratch.** Each cell already holds the code, with blanks marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
import anthropic
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
# Two settings we usually leave alone: retries and the timeout, in seconds.
client = Anthropic(max_retries=2, timeout=30)

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"
QUESTION = "Can I return a jacket after twenty days?"


def ask(client, **extra):
    return client.messages.create(
        model=MODEL,
        max_tokens=100,
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": QUESTION}],
        **extra,
    )

---

## Task 1 - Two families of errors

In [ ]:
# ============================================================
# TASK 1 - Two families of errors
# ============================================================
#
# WHAT TO DO
#   Finish attempt(): it runs a function, and when the library
#   raises, it records what kind of failure it was - the class,
#   the status code and the message for a status error, the
#   class alone for a connection error. Try it on a normal
#   request.
#
# WHY IT MATTERS
#   The library raises two families. APIStatusError means the
#   server answered with an error code. APIConnectionError means
#   no answer came at all - so there is no status code to read.
#   Everything about handling failures starts from that split,
#   so catch the two families separately, not "Exception".
#
# WHERE TO SEE IT IN THE LECTURE
#   "Note the two families".
#
# HOW TO DO IT
#   Three blanks:
#
#       anthropic.APIStatusError       the server answered with an error
#       e.status_code                  the code it answered with
#       anthropic.APIConnectionError   no answer at all
# ============================================================

# TODO: replace each ... below with the value named beside it
def attempt(label, fn):
    try:
        fn()
        print(f"{label}: ok")
        return {"label": label, "ok": True, "error": None, "status": None, "message": ""}
    except anthropic.APIStatusError as e:
        message = e.body["error"]["message"]
        print(f"{label}: {type(e).__name__} {e.status_code}\n    {message[:66]}")
        return {"label": label, "ok": False, "error": type(e).__name__,
                "status": ..., "message": message}   # e.status_code
    except ... as e:   # anthropic.APIConnectionError - no answer at all
        print(f"{label}: {type(e).__name__}")
        return {"label": label, "ok": False, "error": type(e).__name__,
                "status": None, "message": str(e)}

attempt("good request", lambda: ask(client))

check("error_families", attempt=attempt)

---

## Task 2 - Make it fail three ways

In [ ]:
# ============================================================
# TASK 2 - Make it fail three ways
# ============================================================
#
# WHAT TO DO
#   Fail on purpose, three times: a client with a wrong key, a
#   parameter current models no longer accept - sent through
#   extra_body, because the library has dropped it - and a
#   50-millisecond timeout with no retries.
#
# WHY IT MATTERS
#   Read what each failure says. 401: the key is wrong - the
#   request fails the same way a thousand times. 400: the
#   request itself is wrong, and it stays wrong. The timeout is
#   different: the request was fine, the answer was slow. Only
#   that one might succeed on another try.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Now the failures".
#
# HOW TO DO IT
#   Three blanks:
#
#       "sk-ant-wrong-key"        a key the API will refuse
#       {"temperature": 0.2}      the removed parameter, in extra_body
#       0.05                      a timeout no model can meet
# ============================================================

try:
    # TODO: replace each ... below with the value named beside it
    bad_key = Anthropic(api_key=...)   # "sk-ant-wrong-key"
    slow = client.with_options(timeout=..., max_retries=0)   # 0.05 - fifty milliseconds
    results = [
        attempt("wrong key", lambda: ask(bad_key)),
        attempt("old parameter", lambda: ask(client, extra_body=...)),   # {"temperature": 0.2}
        attempt("timeout 50 ms", lambda: ask(slow)),
    ]
except TypeError as exc:   # provided: a blank left above or in Task 1, said plainly
    if "catching classes" in str(exc):
        raise ValueError("attempt() from Task 1 still has a ... in its except clause. "
                         "Finish Task 1 first.") from None
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

check("failure_runs", results=results)

---

## Task 3 - Retry only what can change

In [ ]:
# ============================================================
# TASK 3 - Retry only what can change
# ============================================================
#
# WHAT TO DO
#   Write should_retry(): True for the failures that can pass on
#   their own - 429, 5xx, timeouts, lost connections - and False
#   for the rest. Then watch the library do the retrying: an
#   overloaded API answers 529 twice; max_retries=2 rides it out,
#   max_retries=0 does not.
#
# WHY IT MATTERS
#   It is tempting to catch every exception and retry. Then a
#   wrong key becomes an endless loop of 401s that costs time and
#   hides the real problem. 4xx are your bug: fix the request.
#   429, 5xx and timeouts are the weather: retry with a pause -
#   which max_retries already does. Raise it for a batch job; set
#   it to zero when a person is waiting.
#
# WHERE TO SEE IT IN THE LECTURE
#   "That is the split that matters".
#
# HOW TO DO IT
#   Three blanks:
#
#       (408, 409, 429)                the client-side codes worth retrying
#       500                            and everything from here up
#       anthropic.APIConnectionError   timeouts and lost connections
# ============================================================

try:
    # TODO: replace each ... below with the value named beside it
    def should_retry(exc):
        if isinstance(exc, ...):   # anthropic.APIConnectionError
            return True
        code = getattr(exc, "status_code", 0)
        return code in ... or code >= ...   # (408, 409, 429), 500

    shopassist_lab.simulate_overload(times=2)   # provided: the API answers 529 twice
    result = attempt("overloaded, max_retries=2", lambda: ask(client))
    recovered = dict(result, attempts=shopassist_lab.ATTEMPTS[-1])
    print("   attempts:", recovered["attempts"])

    shopassist_lab.simulate_overload(times=1)
    failed = attempt("overloaded, max_retries=0", lambda: ask(client.with_options(max_retries=0)))
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

check("retry_policy", should_retry=should_retry, recovered=recovered, failed=failed)

---

## Done

4xx are your bug: fix the request. 429, 5xx and timeouts are the weather: retry with a pause - and let the library's max_retries do that for you. Catch the two families separately, and never retry a request that cannot change.